In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG        = "clutchlytics"
BRONZE_TABLE   = f"{CATALOG}.bronze.raw_nhl_skater_logs"
DIM_ATHLETES   = f"{CATALOG}.silver.dimAthletes"
DIM_GAMES      = f"{CATALOG}.silver.dimGames"
SILVER_TABLE   = f"{CATALOG}.silver.nhl_skater_game_logs"
 
LEAGUE         = "nhl"
SEASON         = 2026
 
# True  → reprocess all 424 athletes (use for first run or schema changes)
# False → incremental, only athletes where Bronze is newer than Silver
FULL_REFRESH   = True
 
print(f"Source       : {BRONZE_TABLE}")
print(f"dimAthletes  : {DIM_ATHLETES}")
print(f"dimGames     : {DIM_GAMES}")
print(f"Target       : {SILVER_TABLE}")
print(f"League       : {LEAGUE}")
print(f"Season       : {SEASON}")
print(f"Full refresh : {FULL_REFRESH}")

In [0]:
# ── DETERMINE ATHLETES TO PROCESS ────────────────────────────────────────────
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone
import json
 
bronze_df = spark.table(BRONZE_TABLE)
 
if not FULL_REFRESH and spark.catalog.tableExists(SILVER_TABLE):
    silver_latest = spark.sql(f"""
        SELECT athlete_id, MAX(silver_ingested_at) AS last_updated
        FROM {SILVER_TABLE}
        GROUP BY athlete_id
    """)
    to_process = (
        bronze_df
        .join(silver_latest, on="athlete_id", how="left")
        .filter(
            F.col("last_updated").isNull() |
            (F.col("pulled_at") > F.col("last_updated"))
        )
        .drop("last_updated")
    )
    print(f"Incremental — athletes to process: {to_process.count()}")
else:
    to_process = bronze_df
    print(f"Full refresh — athletes to process: {to_process.count()}")

In [0]:
# ── LOAD DIM REFERENCES ───────────────────────────────────────────────────────
 
# dimAthletes — current active rows only
dim_athletes = (
    spark.table(DIM_ATHLETES)
    .filter(
        (F.col("league") == LEAGUE) &
        (F.col("is_current") == True)
    )
    .select(
        F.col("athlete_id").cast("string").alias("dim_athlete_id"),
        F.col("clutch_athlete_id"),
        F.col("clutch_team_id"),
        F.col("position_abbr"),
    )
)
 
# dimGames — join on source_event_id + league to resolve clutch_game_id
# Alias conflicting columns to avoid ambiguity
dim_games = (
    spark.table(DIM_GAMES)
    .filter(F.col("league") == LEAGUE)
    .select(
        F.col("source_event_id").alias("dim_event_id"),
        F.col("clutch_game_id"),
        F.col("round").alias("dim_round"),
        F.col("is_playoff").alias("dim_is_playoff"),
        F.col("game_number_in_series"),
        F.col("series_key"),
    )
)
 
print(f"dimAthletes (active, {LEAGUE}) : {dim_athletes.count()}")
print(f"dimGames ({LEAGUE})            : {dim_games.count()}")

In [0]:
# ── UNPACK GAMELOG ROWS ───────────────────────────────────────────────────────
# Core transform — one Bronze row per athlete → many Silver rows (one per game).
# Runs on driver — necessary for Python-level JSON unpacking of stats arrays.
 
silver_ingested_at = datetime.now(timezone.utc).isoformat()
rows               = []
athletes_processed = 0
athletes_skipped   = 0
 
bronze_rows = to_process.collect()
 
for br in bronze_rows:
    athlete_id   = br["athlete_id"]
    athlete_name = br["athlete_name"]
    team_abbr    = br["team_abbreviation"]
    team_id      = br["team_id"]
    pulled_at    = br["pulled_at"]
    source_file  = br["source_file"]
    meta_season  = br["season"]
 
    # Skip if meta season doesn't match target season
    if str(meta_season) != str(SEASON):
        athletes_skipped += 1
        continue
 
    # ── Parse names array ──
    names_list = br["names"].split("|") if br["names"] else []
    if not names_list:
        athletes_skipped += 1
        print(f"  SKIPPED (no names): {athlete_name} ({athlete_id})")
        continue
 
    # Drop 'production' field — intentionally excluded
    # Find its index so we can skip it during stat extraction
    prod_idx = names_list.index("production") if "production" in names_list else None
 
    # ── Unpack seasonTypes JSON ──
    try:
        season_types = json.loads(br["season_types_json"])
    except Exception as e:
        athletes_skipped += 1
        print(f"  SKIPPED (JSON parse error): {athlete_name} ({athlete_id}): {e}")
        continue
 
    for st in season_types:
        for cat in st.get("categories", []):
            for event in cat.get("events", []):
                event_id   = event.get("eventId")
                stats      = event.get("stats", [])
                event_type = event.get("type", {})
                type_abbr  = event_type.get("abbreviation", "")
 
                # ── Determine season_type from event type abbreviation ──
                # RD* = playoff round, STD = regular season standard
                if type_abbr.startswith("RD") or "round" in event_type.get("slug", ""):
                    season_type_label = "playoffs"
                    round_label       = br["round"]
                else:
                    season_type_label = "regular"
                    round_label       = None
 
                # ── Zip names with stats → named dict ──
                stat_dict = {}
                for i, name in enumerate(names_list):
                    if name == "production":
                        continue   # drop production field
                    stat_dict[name] = stats[i] if i < len(stats) else None
 
                # ── Helper: safe stat cast ──
                def to_int(val):
                    try:
                        return int(float(val)) if val not in (None, "", "null") else None
                    except:
                        return None
 
                def to_float(val):
                    try:
                        return float(val) if val not in (None, "", "null") else None
                    except:
                        return None
 
                def parse_toi(toi_str):
                    """Convert MM:SS string to integer seconds."""
                    try:
                        if not toi_str or ":" not in str(toi_str):
                            return 0
                        parts = str(toi_str).split(":")
                        return int(parts[0]) * 60 + int(parts[1])
                    except:
                        return 0
 
                # ── Parse TOI ──
                toi_seconds = parse_toi(stat_dict.get("timeOnIcePerGame"))
 
                # ── played_in_game flag ──
                played_in_game = toi_seconds > 0
 
                rows.append({
                    # ── Natural keys ──
                    "athlete_id":           athlete_id,
                    "event_id":             event_id,
 
                    # ── Context ──
                    "athlete_name":         athlete_name,
                    "team_abbreviation":    team_abbr,
                    "team_id":              team_id,
                    "season":               SEASON,
                    "season_type":          season_type_label,
                    "round":                to_int(round_label),
                    "is_playoff":           season_type_label == "playoffs",
 
                    # ── Participation flag ──
                    "played_in_game":       played_in_game,
 
                    # ── Skater stats (typed) ──
                    "goals":                to_int(stat_dict.get("goals")),
                    "assists":              to_int(stat_dict.get("assists")),
                    "points":               to_int(stat_dict.get("points")),
                    "plus_minus":           to_int(stat_dict.get("plusMinus")),
                    "penalty_minutes":      to_int(stat_dict.get("penaltyMinutes")),
                    "shots":                to_int(stat_dict.get("shotsTotal")),
                    "shooting_pct":         to_float(stat_dict.get("shootingPct")),
                    "pp_goals":             to_int(stat_dict.get("powerPlayGoals")),
                    "pp_assists":           to_int(stat_dict.get("powerPlayAssists")),
                    "sh_goals":             to_int(stat_dict.get("shortHandedGoals")),
                    "sh_assists":           to_int(stat_dict.get("shortHandedAssists")),
                    "game_winning_goals":   to_int(stat_dict.get("gameWinningGoals")),
                    "toi_seconds":          toi_seconds,
 
                    # ── Metadata ──
                    "silver_ingested_at":   silver_ingested_at,
                    "source_file":          source_file,
                    "pulled_at":            pulled_at,
                })
 
    athletes_processed += 1
 
print(f"\nAthletes processed : {athletes_processed}")
print(f"Athletes skipped   : {athletes_skipped}")
print(f"Game rows built    : {len(rows)}")

In [0]:
# ── SPOT CHECK — verify one athlete's rows ────────────────────────────────────
 
if rows:
    sample_athlete = rows[0]["athlete_name"]
    sample_rows    = [r for r in rows if r["athlete_name"] == sample_athlete]
 
    print(f"Spot check — {sample_athlete} ({sample_rows[0]['athlete_id']})")
    print(f"Total game rows   : {len(sample_rows)}")
    print(f"Playoff rows      : {sum(1 for r in sample_rows if r['is_playoff'])}")
    print(f"Regular season    : {sum(1 for r in sample_rows if not r['is_playoff'])}")
    print(f"played_in_game=T  : {sum(1 for r in sample_rows if r['played_in_game'])}")
    print(f"\nSample game row:")
    for k, v in sample_rows[0].items():
        print(f"  {k:<25} = {v}")

In [0]:
# ── BUILD DATAFRAME + JOIN DIM REFERENCES ────────────────────────────────────────────────────
 
if not rows:
    raise ValueError("No rows built — check parsing above before writing.")
 
silver_df = spark.createDataFrame(rows)
 
# ── Join dimAthletes → clutch_athlete_id ──
silver_df = (
    silver_df
    .join(
        dim_athletes,
        silver_df.athlete_id == dim_athletes.dim_athlete_id,
        how="left"
    )
    .drop("dim_athlete_id")
)
 
# ── Join dimGames → clutch_game_id ──
# Regular season games will be NULL — dimGames only has playoff games currently
silver_df = (
    silver_df
    .join(
        dim_games,
        silver_df.event_id == dim_games.dim_event_id,
        how="left"
    )
    .drop("dim_event_id")
)
 
# ── Warn on unmatched athlete joins ──
unmatched_athletes = silver_df.filter(F.col("clutch_athlete_id").isNull()).select(
    "athlete_id", "athlete_name"
).distinct()
unmatched_count = unmatched_athletes.count()
print(f"Unmatched athletes (no clutch_athlete_id): {unmatched_count}")
if unmatched_count > 0:
    print("Sample unmatched:")
    unmatched_athletes.show(10, truncate=False)
 
# ── Final column selection and ordering ──
silver_df = silver_df.select(
    # ── Surrogate FKs ──
    "clutch_athlete_id",
    "clutch_game_id",
 
    # ── Natural keys ──
    "athlete_id",
    "event_id",
 
    # ── Context ──
    "athlete_name",
    "team_abbreviation",
    "team_id",
    "season",
    "season_type",
    F.coalesce(F.col("dim_round"), F.col("round")).alias("round"),
    F.coalesce(F.col("dim_is_playoff"), F.col("is_playoff")).alias("is_playoff"),
    "game_number_in_series",
    "series_key",
 
    # ── Participation ──
    "played_in_game",
 
    # ── Stats ──
    "goals",
    "assists",
    "points",
    "plus_minus",
    "penalty_minutes",
    "shots",
    "shooting_pct",
    "pp_goals",
    "pp_assists",
    "sh_goals",
    "sh_assists",
    "game_winning_goals",
    "toi_seconds",
 
    # ── Metadata ──
    "silver_ingested_at",
    "source_file",
    "pulled_at",
)
 
total_rows = silver_df.count()
print(f"Total Silver rows to write: {total_rows}")

In [0]:
# ── WRITE TO SILVER ───────────────────────────────────────────────────────────
# First run: create table.
# Re-run (same athletes): MERGE on athlete_id + event_id.
# New athletes added later: MERGE inserts them cleanly.
 
table_exists = spark.catalog.tableExists(SILVER_TABLE)
 
if not table_exists:
    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Table created: {SILVER_TABLE}")
 
else:
    silver_df.createOrReplaceTempView("new_skater_logs")
 
    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING new_skater_logs AS source
        ON  target.athlete_id = source.athlete_id
        AND target.event_id   = source.event_id
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)
    print(f"Merged into existing table: {SILVER_TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
print("── Playoff rows sample ──")
spark.sql(f"""
    SELECT
        athlete_name,
        team_abbreviation,
        event_id,
        clutch_game_id,
        season_type,
        round,
        game_number_in_series,
        played_in_game,
        goals,
        assists,
        points,
        toi_seconds,
        shots,
        plus_minus
    FROM {SILVER_TABLE}
    WHERE is_playoff = true
    ORDER BY athlete_name, event_id
    LIMIT 30
""").show(30, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                        AS total_rows,
        COUNT(DISTINCT athlete_id)                                      AS unique_athletes,
        COUNT(DISTINCT event_id)                                        AS unique_games,
        COUNT(CASE WHEN is_playoff = true  THEN 1 END)                 AS playoff_rows,
        COUNT(CASE WHEN is_playoff = false THEN 1 END)                 AS regular_rows,
        COUNT(CASE WHEN played_in_game = true  THEN 1 END)             AS played_rows,
        COUNT(CASE WHEN played_in_game = false THEN 1 END)             AS scratched_rows,
        COUNT(CASE WHEN clutch_athlete_id IS NULL THEN 1 END)          AS unmatched_athletes,
        COUNT(CASE WHEN clutch_game_id IS NULL
                    AND is_playoff = true THEN 1 END)                  AS unmatched_playoff_games,
        COUNT(CASE WHEN clutch_game_id IS NULL
                    AND is_playoff = false THEN 1 END)                 AS expected_null_reg_games,
        COUNT(CASE WHEN toi_seconds IS NULL THEN 1 END)                AS null_toi,
        COUNT(CASE WHEN goals IS NULL THEN 1 END)                      AS null_goals,
        ROUND(AVG(CASE WHEN played_in_game THEN toi_seconds END), 0)   AS avg_toi_seconds,
        SUM(CASE WHEN is_playoff THEN goals ELSE 0 END)                AS total_playoff_goals,
        SUM(CASE WHEN is_playoff THEN points ELSE 0 END)               AS total_playoff_points
    FROM {SILVER_TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)